# Functions, comprehensions, generators

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~35 min

## def, return, arguments

A function names a computation so you can reuse and test it. `def` creates it; `return` hands back the result and exits immediately — code after `return` never runs. Arguments can have defaults; callers may pass them by name (`f(x, scale=2)`).



In [ ]:
def accuracy(y_true, y_pred):
    """Fraction of correct predictions."""   # docstring: the function's one-line manual
    correct = sum(t == p for t, p in zip(y_true, y_pred))  # True counts as 1
    return correct / len(y_true)

accuracy([1, 0, 1], [1, 1, 1])     # 2/3 ≈ 0.667

def clip(x, lo=0.0, hi=1.0):       # defaults make arguments optional
    return max(lo, min(x, hi))
clip(1.7), clip(-0.2), clip(0.4)   # (1.0, 0.0, 0.4)

A function that reaches the end without `return` hands back `None` — the source of "TypeError: unsupported operand" surprises two cells later. Variables created inside are **local**: they vanish when the function ends, and a same-named variable outside is untouched.

> **Note.** One famous trap: a default value is created **once**, when `def` runs — `def f(x=[])` shares that single list across every call, so `f(); f()` returns `[1, 1]`, not `[1]`. Default to `x=None` and build the list inside the function.



## Comprehensions

A comprehension builds a container in one expression: `[expr for x in xs]` maps every item, adding `if cond` at the end filters first. It replaces the append-in-a-loop pattern — shorter to write, faster to run. For dicts, looping yields the **keys** (`for k in d`), `.items()` yields `(key, value)` pairs, and `{k: v for ...}` builds a dict the same way.



In [ ]:
scores = [0.71, 0.85, 0.64, 0.92]
squares = [s * s for s in scores]                    # map: every item transformed
high    = [s for s in scores if s > 0.8]             # filter: keep some items
rounded = [round(s, 1) if s > 0.7 else 0 for s in scores]  # if-else in the VALUE slot
label2id = {"cat": 0, "dog": 1}
id2label = {v: k for k, v in label2id.items()}       # dict comprehension — swap pairs
uniq_len = {len(name) for name in label2id}          # set comprehension → {3}

Two different `if`s: `if cond` at the **end** decides which items appear; `a if cond else b` in the **value slot** picks between two values per item. Comprehensions nest (`[(i, j) for i in a for j in b]`), but once one needs a comment to explain it, write the loop — readers parse loops faster.



## Generators: lazy sequences

A **generator expression** is a comprehension in round brackets: it produces items **on demand** instead of building a list in memory. For a billion items that is the difference between running and crashing.



In [ ]:
xs = range(10)
list_comp = [x * x for x in xs]   # a real list — 10 slots in memory
gen       = (x * x for x in xs)   # a generator — nothing computed yet
sum(gen)                          # 285 — pulls items one at a time
sum(gen)                          # 0 — generators are single-use, consumed forever

`def` + `yield` makes a custom generator: the function pauses at each `yield` and resumes where it left off on the next pull. `for` pulls items one by one:



In [ ]:
def batches(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]      # hand out one chunk, pause here

for chunk in batches(range(10), 4):
    print(chunk)                   # [0,1,2,3] then [4,5,6,7] then [8,9]

> **Note.** That is exactly what a DataLoader does — yield one batch at a time instead of materializing all batches. When a contest file is too big for memory, loop over its lines or read it with pandas `chunksize`: same lazy idea.



## The iterator toolbox

A few built-ins replace most handwritten loops. `sorted`, `min`, `max` all take `key=` — a function extracting the value to compare by; `lambda x: expr` is a one-expression anonymous function, handy as that key. `any`/`all` ask "does at least one / do all" over lazily produced conditions.



In [ ]:
words = ["pear", "fig", "banana"]
sorted(words, key=len)              # ['fig', 'pear', 'banana'] — sorted by length
max(scores), min(scores)            # (0.92, 0.64)
pairs = [("cat", 0.9), ("dog", 0.7), ("bird", 0.8)]
max(pairs, key=lambda p: p[1])      # ('cat', 0.9) — argmax over pairs

any(s > 0.9 for s in scores)        # True — at least one qualifies
all(s > 0.5 for s in scores)        # True — every one qualifies
sum(s for s in scores if s > 0.8)   # generator inside: no list is ever built

`lambda` is fine for `key=` and a smell anywhere bigger — give real logic a `def` and a name. `reversed(xs)` walks backwards; `itertools` (chain, product, combinations) covers the rest.



### ✏️ g = (x*x for x in range(3)); what does list(g) return after sum(g) already ran?

**A.** [0, 1, 4, 0, 1, 4]
**B.** TypeError
**C.** []
**D.** [0, 1, 4]

<details><summary>Answer</summary>

**C.** []

Generators are consumed once — sum(g) pulled every item, so nothing is left.

</details>

### ✏️ def f(x=[]): x.append(1); return x. What does the second call f() return?

**A.** [1]
**B.** []
**C.** [1, 1]
**D.** An error

<details><summary>Answer</summary>

**C.** [1, 1]

Default arguments are created once at def time and shared by every call. Use x=None as the default.

</details>

### ✏️ Which builds a dict mapping each word to its length?

**A.** dict(words, key=len)
**B.** {w: len(w) for w in words}
**C.** [len(w) for w in words]
**D.** {len(w): w for w in words}

<details><summary>Answer</summary>

**B.** {w: len(w) for w in words}

Key first, colon, value — a dict comprehension. The swapped version maps lengths to words and loses duplicates.

</details>

---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Split into batches

●●○ · ~10 min

> Mini-batching is how every olympiad notebook iterates data. `np.array_split` handles the ragged last batch for you — implement it once by hand so you understand the indexing.

Implement `batches(n, bs)` → list of `(start, end)` index pairs covering `0..n` in chunks of at most `bs`. Example: `batches(10, 4)` → `[(0,4),(4,8),(8,10)]`.

<details><summary>Hint</summary>

list(range(0, n, bs)) gives the starts; each end is min(start + bs, n).

</details>


In [ ]:
import numpy as np

def batches(n, bs):
    # your code
    return []

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    return {\"n\": int(rng.integers(1, 60)), \"bs\": int(rng.integers(1, 12))}\n\ndef batches(n, bs):\n    return [(s, min(s + bs, n)) for s in range(0, n, bs)]", "batches", 20)

### Task 2 — Word n-grams

●●○ · ~8 min

> Word n-grams capture local context that bag-of-words loses ('not good' ≠ 'good'). They are just sliding windows over the token list — and the building block of every statistical language model.

Implement `ngrams(tokens, n)` → list of tuples of `n` consecutive tokens. `ngrams(['a','b','c'], 2)` → `[('a','b'), ('b','c')]`. If `n` exceeds the number of tokens, return an empty list.

<details><summary>Hint</summary>

[tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)] — the range is empty when the bound is non-positive.

</details>


In [ ]:
import numpy as np

def ngrams(tokens, n):
    # your code
    return []

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    toks = [str(w) for w in rng.choice([\"a\", \"b\", \"c\", \"d\", \"e\", \"f\"],\n                                      int(rng.integers(3, 12)))]\n    return {\"tokens\": toks, \"n\": int(rng.integers(1, 5))}\n\ndef ngrams(tokens, n):\n    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]", "ngrams", 20)

---

**Go deeper**
- [The Python Tutorial (official)](https://docs.python.org/3/tutorial/)
- [CS231n: Python & NumPy tutorial](https://cs231n.github.io/python-numpy-tutorial/)